# SegCT/MRI

Automatic anatomical segmentation from CT/MRI using **TotalSegmentator**. Lite and Local use this same notebook. Compatible DICOM series are converted to NIfTI inside SegRef3D with their patient geometry preserved.

1. Select **Runtime > Change runtime type > T4 GPU** (recommended).
2. Choose **Runtime > Run all**. The first code cell waits for your ZIP.
3. Upload one SegRef3D request ZIP (for example `segct_mri_request.zip`). Source-prefixed filenames and older request ZIPs are accepted.
4. After upload, setup, validation and segmentation run in sequence without another file prompt.
5. The final cell starts downloading `segct_mri_result.zip`; import it into SegRef3D.

Keep this browser tab open for the final download; your browser may require permission to save it. The ZIP contains medical images: confirm that your institution permits uploading them to Google Colab. Review and refine segmentation results before use.


In [ ]:
#@title 1. Upload SegCT/MRI request ZIP
from google.colab import files
from pathlib import Path
import json, tempfile, zipfile

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(uploaded) != 1 or len(zip_names) != 1:
    raise ValueError('Upload exactly one SegRef3D SegCT/MRI request ZIP, then run all cells again.')
# Keep the upload outside the repository and every setup/work directory.
upload_dir = Path(tempfile.mkdtemp(prefix='segct-mri-upload-', dir='/content'))
upload_name = Path(zip_names[0].replace('\\', '/')).name
REQUEST_ZIP = str(upload_dir / upload_name)
Path(REQUEST_ZIP).write_bytes(uploaded[zip_names[0]])
if not zipfile.is_zipfile(REQUEST_ZIP):
    raise ValueError('The uploaded file is not a valid ZIP archive. Export a new SegCT/MRI request ZIP from SegRef3D.')
try:
    with zipfile.ZipFile(REQUEST_ZIP) as archive:
        info = archive.getinfo('manifest.json')
        if info.file_size > 1024 * 1024:
            raise ValueError('The request manifest is too large.')
        request = json.loads(archive.read(info))
        if (not isinstance(request, dict)
                or request.get('schema') not in ('segref3d-segct-mri-bridge', 'segref3d-instant3d-bridge')
                or request.get('schema_version') != '1.0'
                or not request.get('objects')):
            raise ValueError('This is not a supported SegRef3D SegCT/MRI request. Export it from SegCT/MRI, not SegAnything or Project ZIP.')
        if not {'image/source.nii', 'image/source.nii.gz'}.intersection(archive.namelist()):
            raise ValueError('The request ZIP is missing its source NIfTI volume.')
except (KeyError, json.JSONDecodeError, UnicodeDecodeError, zipfile.BadZipFile) as exc:
    raise ValueError('The request ZIP has a missing or damaged manifest/source. Export it again from SegCT/MRI.') from exc
print('SegCT/MRI request uploaded:', upload_name)
print('Run all continues with setup and segmentation. The upload is kept in:', upload_dir)


In [ ]:
#@title 2. Setup SegCT/MRI
import os, sys, subprocess
from pathlib import Path
if 'REQUEST_ZIP' not in globals() or not Path(REQUEST_ZIP).is_file():
    raise FileNotFoundError('Upload a SegRef3D request ZIP in the first cell before running setup.')
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'TotalSegmentator', 'nibabel', 'numpy', 'Pillow', 'scipy'], check=True)
REPO = '/content/SegRef3D'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/SatoruMuro/SegRef3D.git', REPO], check=True)
sys.path.insert(0, os.path.join(REPO, 'ColabNotebooks'))
sys.path.insert(0, os.path.join(REPO, 'SegRef3D'))
print('Setup complete. GPU runtime is recommended.')
if 'REQUEST_ZIP' not in globals() or not Path(REQUEST_ZIP).is_file():
    raise FileNotFoundError('Upload a SegRef3D request ZIP in the first cell before running setup.')


In [ ]:
#@title 3. Run segmentation
if 'REQUEST_ZIP' not in globals() or not Path(REQUEST_ZIP).is_file():
    raise FileNotFoundError('The uploaded request ZIP is missing. Run the upload cell again.')
import tempfile
from instant3d_bridge import validate_request_zip
with tempfile.TemporaryDirectory() as folder:
    manifest, source_path = validate_request_zip(REQUEST_ZIP, folder)
print('Request ID:', manifest['request_id'])
print('Source:', manifest['source']['shape'], manifest['source']['voxel_spacing_mm'], manifest['source']['orientation'])
print('Objects:')
for item in manifest['objects']:
    print(f"  Obj {item['object_id']}: {item['display_name']}")
from instant3dweb2_backend import process_request
RESULT_ZIP = str(process_request(REQUEST_ZIP, '/content/segct_mri_result.zip'))
print('SegCT/MRI processing complete:', RESULT_ZIP)


## Result contents

The result ZIP contains geometry-preserving binary NIfTI masks, a merged labelmap NIfTI, SegRef3D-compatible label PNGs, `volumes.csv`, and a reproducibility manifest. Lower object IDs have priority only in the merged labelmap when ROIs overlap; individual binary masks preserve overlaps.


In [ ]:
#@title 4. Generate / Download result ZIP
# The segmentation cell generates the result ZIP; download it here.
from google.colab import files
from pathlib import Path
if 'RESULT_ZIP' not in globals() or not Path(RESULT_ZIP).is_file():
    raise FileNotFoundError('No SegCT/MRI result ZIP exists. Review the setup/segmentation error above before retrying.')
files.download(RESULT_ZIP)
